# Geometric Brownian Motion (GBM) & Lognormal Stock Simulation

This notebook provides numerical simulations and visualizations of stock price dynamics under Geometric Brownian Motion (GBM) with continuous dividend yields ($q$), supporting the derivations in **Part 2: Deriving the Black-Scholes-Merton Equation**.

We use **Matplotlib** for all charts to ensure they are fully pre-rendered and visible when viewed directly on GitHub.

In [ ]:
import numpy as np
from scipy.stats import norm, lognorm
import matplotlib.pyplot as plt

# Set plot styling for professional look
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.titlesize': 14
})
print("Libraries loaded successfully.")

## 1. SDE Path Simulation

We simulate stock price paths under the SDE:
$$dS_t = (\mu - q)S_t\,dt + \sigma S_t\,dW_t$$

Using the exact integration solution:
$$S_{t+\Delta t} = S_t \exp\left( \left(\mu - q - \frac{1}{2}\sigma^2\right)\Delta t + \sigma \sqrt{\Delta t} Z_t \right)$$
where $Z_t \sim \mathcal{N}(0, 1)$.

In [ ]:
# Parameters
S0 = 100.0
mu = 0.08
q = 0.02
sigma = 0.20
T = 1.0
N_steps = 252  # Trading days in a year
N_paths = 100

dt = T / N_steps
np.random.seed(42)

# Simulate paths
paths = np.zeros((N_steps + 1, N_paths))
paths[0] = S0

for t in range(1, N_steps + 1):
    Z = np.random.normal(size=N_paths)
    paths[t] = paths[t-1] * np.exp((mu - q - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)

# Find max, min and median path indices based on terminal stock price
terminal_prices = paths[-1]
idx_max = np.argmax(terminal_prices)
idx_min = np.argmin(terminal_prices)
idx_med = np.argmin(np.abs(terminal_prices - np.median(terminal_prices)))

# Plot paths
plt.figure(figsize=(10, 5.5))

# Plot all 100 paths in light gray
for i in range(N_paths):
    if i == 0:
        plt.plot(paths[:, i], color='lightgray', alpha=0.4, linewidth=0.8, label='Simulated Paths')
    else:
        plt.plot(paths[:, i], color='lightgray', alpha=0.4, linewidth=0.8)

# Highlight special paths
plt.plot(paths[:, idx_max], color='crimson', linewidth=2.5, label='Maximum Path')
plt.plot(paths[:, idx_min], color='dodgerblue', linewidth=2.5, label='Minimum Path')
plt.plot(paths[:, idx_med], color='goldenrod', linewidth=2.5, label='Median Path')

plt.title(fr"GBM Simulated Stock Price Paths ({N_paths} Paths, $\mu={mu:.2f}$, $q={q:.2f}$, $\sigma={sigma:.2f}$)")
plt.xlabel("Trading Steps (Days)")
plt.ylabel(r"Stock Price ($S_t$)")
plt.xlim(0, N_steps)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()


## 2. Terminal Price Distribution vs. Lognormal PDF

We simulate $100,000$ paths to verify that the simulated terminal stock prices ($S_T$) converge exactly to the theoretical **lognormal probability density function**:
$$f(S_T) = \frac{1}{S_T \sigma \sqrt{2\pi T}} \exp\left( -\frac{\left(\ln(S_T/S_0) - \left(\mu - q - \frac{1}{2}\sigma^2\right)T\right)^2}{2\sigma^2 T} \right)$$

In [ ]:
N_large = 100_000
np.random.seed(42)

# Direct terminal price simulation
Z_large = np.random.normal(size=N_large)
S_T = S0 * np.exp((mu - q - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z_large)

# Analytical PDF range
S_range = np.linspace(30, 220, 500)
mu_l = np.log(S0) + (mu - q - 0.5 * sigma**2) * T
sigma_l = sigma * np.sqrt(T)
pdf_theoretical = lognorm.pdf(S_range, s=sigma_l, scale=np.exp(mu_l))

# Plot histogram vs theoretical curve
plt.figure(figsize=(10, 5.5))
plt.hist(S_T, bins=100, density=True, alpha=0.6, color='steelblue', edgecolor='white', label='Simulated Terminal Prices')
plt.plot(S_range, pdf_theoretical, color='darkorange', linewidth=2.5, label='Analytical Lognormal PDF')
plt.title(r"Terminal Price Distribution vs. Analytical Lognormal PDF")
plt.xlabel(r"Terminal Stock Price ($S_T$)")
plt.ylabel("Probability Density")
plt.legend()
plt.xlim(30, 220)
plt.tight_layout()
plt.show()


## 3. Log-Returns are Normal

The log stock price return $Y = \ln(S_T / S_0)$ is normally distributed:
$$Y = \ln\left(\frac{S_T}{S_0}\right) \sim \mathcal{N}\left( \left(\mu - q - \frac{1}{2}\sigma^2\right)T, \, \sigma^2 T \right)$$

We verify this by plotting the histogram of $\ln(S_T / S_0)$ and overlaying a normal bell curve.

In [ ]:
log_returns = np.log(S_T / S0)

# Normal parameters
mean_r = (mu - q - 0.5 * sigma**2) * T
std_r = sigma * np.sqrt(T)

lr_range = np.linspace(mean_r - 4 * std_r, mean_r + 4 * std_r, 500)
pdf_normal = norm.pdf(lr_range, loc=mean_r, scale=std_r)

# Plot log-returns
plt.figure(figsize=(10, 5.5))
plt.hist(log_returns, bins=100, density=True, alpha=0.6, color='forestgreen', edgecolor='white', label='Simulated Log-Returns')
plt.plot(lr_range, pdf_normal, color='crimson', linewidth=2.5, label='Analytical Normal PDF')
plt.title("Log-Returns Distribution vs. Analytical Normal PDF")
plt.xlabel(r"Log-Return $\ln(S_T / S_0)$")
plt.ylabel("Probability Density")
plt.legend()
plt.xlim(mean_r - 4 * std_r, mean_r + 4 * std_r)
plt.tight_layout()
plt.show()


## 4. Parameter Sensitivity (Volatility, Maturity, and Dividends)

We visualize how changes in $\sigma$, $T$, and $q$ deform the probability density curve.

In [ ]:
S_grid = np.linspace(10, 300, 1000)
# Use vertical subplots for maximum readability
fig, axes = plt.subplots(3, 1, figsize=(10, 14))

# 1. Volatility Effect
for sig in [0.10, 0.20, 0.35, 0.50]:
    m = np.log(S0) + (mu - q - 0.5 * sig**2) * T
    s = sig * np.sqrt(T)
    axes[0].plot(S_grid, lognorm.pdf(S_grid, s=s, scale=np.exp(m)), label=fr"$\sigma={sig:.2f}$")
axes[0].set_title(r"Effect of Volatility ($\sigma$) on Lognormal PDF")
axes[0].set_xlabel(r"$S_T$")
axes[0].set_ylabel("Density")
axes[0].legend()
axes[0].set_xlim(20, 260)

# 2. Maturity Effect
for mat in [0.25, 0.5, 1.0, 2.0]:
    m = np.log(S0) + (mu - q - 0.5 * sigma**2) * mat
    s = sigma * np.sqrt(mat)
    axes[1].plot(S_grid, lognorm.pdf(S_grid, s=s, scale=np.exp(m)), label=fr"$T={mat:.2f}$ yr")
axes[1].set_title(r"Effect of Maturity ($T$) on Lognormal PDF")
axes[1].set_xlabel(r"$S_T$")
axes[1].set_ylabel("Density")
axes[1].legend()
axes[1].set_xlim(20, 260)

# 3. Dividend Yield Effect
for div in [0.00, 0.04, 0.08, 0.12]:
    m = np.log(S0) + (mu - div - 0.5 * sigma**2) * T
    s = sigma * np.sqrt(T)
    axes[2].plot(S_grid, lognorm.pdf(S_grid, s=s, scale=np.exp(m)), label=fr"$q={div:.2f}$")
axes[2].set_title(r"Effect of Dividends ($q$) on Lognormal PDF")
axes[2].set_xlabel(r"$S_T$")
axes[2].set_ylabel("Density")
axes[2].legend()
axes[2].set_xlim(20, 260)

plt.tight_layout()
plt.show()
